# 앙상블 가중치 비교 & 비트코인 가격 예측 적중률 분석

## 목적
1. **FinBERT:Gemma 가중치 5가지(0:100 ~ 100:0)** 중 감정 분석에 가장 적합한 비율 탐색
2. 뉴스 감성 분석 결과로 **비트코인 가격 방향(상승/하락) 예측 적중률** 수치 확인

## 노트북 구성
1. 환경 설정
2. 데이터 로드 (뉴스 + 가격)
3. FinBERT + Gemma 개별 분석
4. 가중치별 앙상블 점수 계산 (Gemma 미실행 시 자동 스킵)
5. 가중치별 분포 비교
6. 가격 방향 예측 준비
7. 가중치별 예측 적중률 계산
8. 종합 시각화 (히트맵)
9. 최적 가중치 결론
10. 전체 적중률 요약


---
## 1. 환경 설정


In [ ]:
# ── 표준 라이브러리 ──────────────────────────────────────────────────────
import os
import sys
import time
import warnings
from pathlib import Path

# ── 서드파티 ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv
from scipy import stats

# ── matplotlib 한글 폰트 설정 (Windows: Malgun Gothic) ──────────────────
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

# ── 프로젝트 루트를 sys.path 에 추가 ────────────────────────────────────
PROJECT_ROOT = Path('../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

DATA_RAW  = PROJECT_ROOT / 'data' / 'raw'
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'

print(f'프로젝트 루트: {PROJECT_ROOT}')


---
## 2. 데이터 로드

뉴스 감성 데이터(news_sentiment.csv)와 가격-감성 연동 데이터(merged_analysis.csv)를 로드한다.


In [ ]:
# ── 뉴스 감성 데이터 로드 ────────────────────────────────────────────────
SENTIMENT_FILE = DATA_PROC / 'news_sentiment.csv'
MERGED_FILE    = DATA_PROC / 'merged_analysis.csv'

news_df = pd.read_csv(SENTIMENT_FILE, parse_dates=['published_at'])
merged_df = pd.read_csv(MERGED_FILE, parse_dates=['window_start', 'open_time'])

print(f'뉴스 감성 데이터: {len(news_df)}건')
print(f'  컬럼: {list(news_df.columns)}')
print(f'\n가격-감성 병합 데이터: {len(merged_df)}건')
print(f'  컬럼: {list(merged_df.columns)}')
print(f'\n기간: {merged_df["window_start"].min()} ~ {merged_df["window_start"].max()}')


In [ ]:
# ── 가격 데이터 확인 ─────────────────────────────────────────────────────
print('=== 가격-감성 병합 데이터 샘플 ===')
print(merged_df[['window_start', 'window_score', 'return_5m', 'return_15m', 'return_30m', 'return_60m']].head(10).to_string(index=False))

print(f'\n수익률 통계 (단위: %)')
returns_cols = ['return_5m', 'return_15m', 'return_30m', 'return_60m']
print(merged_df[returns_cols].describe().round(4).to_string())


---
## 3. FinBERT + Gemma 개별 분석

뉴스 본문을 포함한 텍스트로 FinBERT와 Gemma를 각각 실행한다.  
**Gemma(Ollama) 미실행 시**: 자동으로 FinBERT 단독 모드로 전환되며,  
섹션 4~5(가중치 비교)는 건너뛰고 섹션 6~10(가격 예측)만 진행한다.


In [ ]:
# ── 감성 분석 모듈 로드 ─────────────────────────────────────────────────
from src.sentiment import (
    load_finbert_pipeline,
    score_headlines,
    check_ollama_available,
    score_headlines_gemma,
    combine_ensemble,
)
import src.sentiment as _sent

# ── 가중치 override 헬퍼 ─────────────────────────────────────────────────
def combine_with_weights(finbert_df, gemma_df, fw, gw):
    """combine_ensemble 을 임시 가중치로 실행"""
    orig_fw, orig_gw = _sent.FINBERT_WEIGHT, _sent.GEMMA_WEIGHT
    _sent.FINBERT_WEIGHT, _sent.GEMMA_WEIGHT = fw, gw
    result = combine_ensemble(finbert_df, gemma_df)
    _sent.FINBERT_WEIGHT, _sent.GEMMA_WEIGHT = orig_fw, orig_gw
    return result


In [ ]:
# ── 분석용 텍스트 준비 (title + body[:512]) ─────────────────────────────
def prepare_texts(df: pd.DataFrame, body_limit: int = 512) -> list[str]:
    texts = []
    for _, row in df.iterrows():
        title = str(row.get('title', '')).strip()
        body  = row.get('body')
        if body and str(body).strip():
            texts.append(f"{title}. {str(body).strip()[:body_limit]}")
        else:
            texts.append(title)
    return texts

texts = prepare_texts(news_df)
print(f'분석 텍스트: {len(texts)}건')
print(f'body 포함: {sum("."+" " in t for t in texts)}건')
print(f'\n샘플: {texts[0][:120]}...')


In [ ]:
# ── FinBERT 분석 실행 ────────────────────────────────────────────────────
print('FinBERT 로드 중...')
nlp = load_finbert_pipeline()
print('FinBERT 분석 중...')
finbert_df = score_headlines(texts, nlp=nlp)
print(f'완료: {len(finbert_df)}건')
print(f'\nFinBERT 점수 통계:')
print(finbert_df['finbert_score'].describe().round(4).to_string())


In [ ]:
# ── Gemma 가용성 확인 및 분석 ───────────────────────────────────────────
gemma_available = check_ollama_available(warmup_timeout=20)
print(f'Ollama/Gemma 사용 가능: {gemma_available}')

if gemma_available:
    GEMMA_SAMPLE_CAP = 30  # CPU 환경에서 현실적인 상한 (약 5분)
    sample_texts = texts[:GEMMA_SAMPLE_CAP]
    print(f'Gemma 분석 중... (전체 {len(texts)}건 중 {len(sample_texts)}건 샘플)')
    gemma_df = score_headlines_gemma(sample_texts, max_total_seconds=360)
    print(f'완료: {len(gemma_df)}건')
    print(f'\nGemma 점수 통계:')
    print(gemma_df['gemma_score'].describe().round(4).to_string())
else:
    gemma_df = None
    print()
    print('⚠️  Ollama 미실행 또는 모델 응답 실패 상태입니다.')
    print('    → 섹션 4~5 (가중치 비교)는 건너뜁니다.')
    print('    → 섹션 6~10 (가격 예측 적중률)은 FinBERT 기준으로 진행합니다.')
    print()
    print('Ollama 실행 방법:')
    print('  1. https://ollama.ai 에서 Ollama 설치')
    print('  2. 터미널에서: ollama pull gemma3:4b')
    print('  3. Ollama 실행 후 이 셀 재실행')

---
## 4. 가중치별 앙상블 점수 계산

FinBERT:Gemma = 0:100, 25:75, 50:50, 75:25, 100:0 의 5가지 비율로 앙상블 점수를 계산한다.  
*(Gemma 미실행 시 이 섹션은 자동으로 건너뜁니다)*


In [ ]:
# ── 가중치별 앙상블 실행 ─────────────────────────────────────────────────
WEIGHT_RATIOS = [
    (0.00, 1.00),  # FinBERT 0%  : Gemma 100%
    (0.25, 0.75),  # FinBERT 25% : Gemma 75%
    (0.50, 0.50),  # FinBERT 50% : Gemma 50%  ← 현재 기본값
    (0.75, 0.25),  # FinBERT 75% : Gemma 25%
    (1.00, 0.00),  # FinBERT 100%: Gemma 0%
]

ensemble_results = {}

if gemma_available and gemma_df is not None:
    for fw, gw in WEIGHT_RATIOS:
        label = f'F{int(fw*100)}:G{int(gw*100)}'
        result = combine_with_weights(finbert_df, gemma_df, fw, gw)
        ensemble_results[label] = result
        print(f'  {label}: 평균 점수={result["ensemble_score"].mean():+.4f}, '
              f'긍정={( result["sentiment_label"]=="positive").sum()}건, '
              f'부정={(result["sentiment_label"]=="negative").sum()}건, '
              f'중립={(result["sentiment_label"]=="neutral").sum()}건')
else:
    # Gemma 미실행: FinBERT 단독 결과만 저장
    result = combine_ensemble(finbert_df, None)
    ensemble_results['F100:G0'] = result
    print('Gemma 미실행 — FinBERT 단독 결과만 저장됨 (F100:G0)')
    print(f'  F100:G0: 평균 점수={result["ensemble_score"].mean():+.4f}')


---
## 5. 가중치별 분포 비교

5가지 가중치 조합의 감정 점수 분포를 히스토그램과 박스플롯으로 비교한다.  
*(Gemma 미실행 시 이 섹션은 자동으로 건너뜁니다)*


In [ ]:
# ── 가중치별 분포 시각화 ─────────────────────────────────────────────────
if not gemma_available or len(ensemble_results) <= 1:
    print('Gemma 미실행 — 가중치 비교 시각화를 건너뜁니다.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('FinBERT:Gemma 가중치별 감정 점수 분포', fontsize=13, fontweight='bold')

    colors = ['#e74c3c', '#e67e22', '#3498db', '#27ae60', '#9b59b6']
    labels_order = ['F0:G100', 'F25:G75', 'F50:G50', 'F75:G25', 'F100:G0']

    # 히스토그램
    ax = axes[0]
    for (label, res), color in zip(
        [(k, ensemble_results[k]) for k in labels_order if k in ensemble_results],
        colors
    ):
        ax.hist(res['ensemble_score'], bins=20, alpha=0.5, label=label, color=color)
    ax.axvline(0, color='gray', linestyle='--')
    ax.set_xlabel('앙상블 감정 점수')
    ax.set_ylabel('기사 수')
    ax.set_title('가중치별 점수 히스토그램')
    ax.legend(fontsize=9)

    # 박스플롯
    ax = axes[1]
    data = [ensemble_results[k]['ensemble_score'].tolist()
            for k in labels_order if k in ensemble_results]
    bp = ax.boxplot(data, labels=[k for k in labels_order if k in ensemble_results],
                   patch_artist=True,
                   medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.axhline(0, color='gray', linestyle='--')
    ax.set_xlabel('FinBERT:Gemma 가중치')
    ax.set_ylabel('앙상블 감정 점수')
    ax.set_title('가중치별 박스플롯')

    plt.tight_layout()
    plt.show()

    # 분포 지표 테이블
    print('\n=== 가중치별 분포 지표 ===')
    rows = []
    for label in labels_order:
        if label not in ensemble_results: continue
        s = ensemble_results[label]['ensemble_score']
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        rows.append({
            '가중치': label,
            '평균': round(s.mean(), 4),
            '표준편차': round(s.std(), 4),
            'IQR': round(q3 - q1, 4),
            '왜도': round(stats.skew(s), 4),
            '첨도': round(stats.kurtosis(s), 4),
        })
    print(pd.DataFrame(rows).set_index('가중치').to_string())


---
## 6. 가격 방향 예측 준비

감성 점수와 실제 BTC 가격 방향(상승/하락)을 연결한다.

- **예측 방향**: `ensemble_score > 0` → 상승 예측, `< 0` → 하락 예측
- **실제 방향**: `return_Xm > 0` → 상승, `< 0` → 하락
- **기준 데이터**: `merged_analysis.csv` (5분 윈도우 집계)


In [ ]:
# ── merged_analysis.csv 기반 예측 준비 ─────────────────────────────────
# merged_df: window_start, window_score, return_5m, return_15m, return_30m, return_60m

ma = merged_df.dropna(subset=['window_score', 'return_5m']).copy()

# 감성 예측 방향: 양수=상승, 음수=하락
ma['pred_direction'] = (ma['window_score'] > 0).astype(int)  # 1=상승예측, 0=하락예측

# 실제 가격 방향
lag_cols = {
    'T+5분':  'return_5m',
    'T+15분': 'return_15m',
    'T+30분': 'return_30m',
    'T+60분': 'return_60m',
}
for name, col in lag_cols.items():
    ma[f'actual_{name}'] = (ma[col] > 0).astype(int)  # 1=실제상승, 0=실제하락

print(f'예측 데이터 건수: {len(ma)}건')
print(f'상승 예측: {ma["pred_direction"].sum()}건 ({ma["pred_direction"].mean()*100:.1f}%)')
print(f'하락 예측: {(1-ma["pred_direction"]).sum()}건 ({(1-ma["pred_direction"]).mean()*100:.1f}%)')
print(f'\n감성 점수 범위: {ma["window_score"].min():.4f} ~ {ma["window_score"].max():.4f}')


---
## 7. 가격 방향 예측 적중률 계산

각 가중치 조합 × 시간 지연(T+5m, T+15m, T+30m, T+60m)의 **예측 적중률** 계산.

- **적중률(Accuracy)**: 예측 방향이 실제 방향과 일치한 비율
- **기준선(Baseline)**: 50% (무작위 예측)
- **Precision / Recall / F1**: 이진 분류 성능 지표


In [ ]:
# ── 적중률 계산 함수 ─────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calc_hit_rate(pred_dir, actual_dir):
    """예측 방향 vs 실제 방향의 이진 분류 지표 반환"""
    p = pred_dir.values
    a = actual_dir.values
    return {
        '적중률(%)': round(accuracy_score(a, p) * 100, 1),
        '정밀도(%)': round(precision_score(a, p, zero_division=0) * 100, 1),
        '재현율(%)': round(recall_score(a, p, zero_division=0) * 100, 1),
        'F1(%)':     round(f1_score(a, p, zero_division=0) * 100, 1),
    }

# ── merged_analysis.csv 기반 적중률 (FinBERT 기준) ─────────────────────
print('=== 감성 분석 → 비트코인 가격 방향 예측 적중률 ===')
print(f'  (기준 데이터: merged_analysis.csv, 총 {len(ma)}개 윈도우)')
print(f'  (기준선: 50% — 무작위 예측 수준)\n')

result_rows = []
for lag_name, col in lag_cols.items():
    metrics = calc_hit_rate(ma['pred_direction'], ma[f'actual_{lag_name}'])
    row = {'시간 지연': lag_name, '실제 상승 비율(%)': round(ma[col].gt(0).mean()*100,1)}
    row.update(metrics)
    result_rows.append(row)

result_df = pd.DataFrame(result_rows).set_index('시간 지연')
print(result_df.to_string())
print(f'\n  → 기준선(50%) 대비 최대 개선: +{(result_df["적중률(%)"] - 50).max():.1f}%p')


In [ ]:
# ── 가중치별 적중률 비교 (Gemma 실행 시) ─────────────────────────────────
# 가중치별로 앙상블 점수를 시간 윈도우에 매핑해 적중률 비교
# merged_analysis.csv의 window_score는 현재 저장된 앙상블 가중치 기준
# → 각 가중치별 재집계가 필요하므로, 뉴스별 점수에서 5분 윈도우를 재집계

if not gemma_available or len(ensemble_results) <= 1:
    print('Gemma 미실행 → 가중치별 비교는 Gemma 실행 후 재분석이 필요합니다.')
    print('현재는 FinBERT 단독(F100:G0) 기준으로만 적중률을 분석합니다.')
    weight_hit_df = None
else:
    # 뉴스 df에 발행 시각 추가
    for label, res in ensemble_results.items():
        res['published_at'] = news_df['published_at'].values

    # 5분 윈도우로 집계
    def agg_to_windows(res_df, weight_label):
        tmp = res_df[['published_at', 'ensemble_score']].copy()
        tmp = tmp.dropna()
        tmp['window_start'] = tmp['published_at'].dt.floor('5min')
        win = tmp.groupby('window_start')['ensemble_score'].mean().reset_index()
        win.columns = ['window_start', f'score_{weight_label}']
        return win

    # 가격 데이터와 조인
    price_base = merged_df[['window_start', 'return_5m', 'return_15m', 'return_30m', 'return_60m']].copy()

    weight_rows = []
    for fw, gw in WEIGHT_RATIOS:
        label = f'F{int(fw*100)}:G{int(gw*100)}'
        if label not in ensemble_results: continue
        win_df = agg_to_windows(ensemble_results[label], label)
        joined = pd.merge(price_base, win_df, on='window_start', how='inner')
        if joined.empty: continue
        pred = (joined[f'score_{label}'] > 0).astype(int)
        for lag_name, col in lag_cols.items():
            actual = (joined[col] > 0).astype(int)
            acc = accuracy_score(actual, pred) * 100
            weight_rows.append({'가중치': label, '시간 지연': lag_name, '적중률(%)': round(acc, 1)})

    if weight_rows:
        weight_hit_df = pd.DataFrame(weight_rows).pivot(index='가중치', columns='시간 지연', values='적중률(%)')
        weight_hit_df = weight_hit_df[list(lag_cols.keys())]
        print('=== 가중치별 × 시간 지연별 예측 적중률 (%) ===')
        print(weight_hit_df.to_string())
        print(f'\n기준선: 50.0%')
    else:
        weight_hit_df = None
        print('데이터 매칭 실패 — 가중치별 비교를 건너뜁니다.')


---
## 8. 종합 시각화

시간 지연별 예측 적중률을 히트맵으로 시각화한다.


In [ ]:
# ── 적중률 시각화 ────────────────────────────────────────────────────────
if weight_hit_df is not None and not weight_hit_df.empty:
    # 가중치별 히트맵
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('FinBERT:Gemma 가중치별 비트코인 방향 예측 적중률', fontsize=13, fontweight='bold')

    ax = axes[0]
    sns.heatmap(
        weight_hit_df,
        annot=True, fmt='.1f', cmap='RdYlGn',
        vmin=40, vmax=70, center=50,
        linewidths=0.5, ax=ax
    )
    ax.set_title('가중치별 × 시간 지연별 적중률 히트맵')
    ax.set_xlabel('시간 지연')
    ax.set_ylabel('FinBERT:Gemma 비율')

    # 막대그래프
    ax = axes[1]
    avg_acc = weight_hit_df.mean(axis=1)
    colors_bar = ['#e74c3c', '#e67e22', '#3498db', '#27ae60', '#9b59b6']
    bars = ax.bar(avg_acc.index, avg_acc.values, color=colors_bar[:len(avg_acc)], alpha=0.8)
    ax.axhline(50, color='gray', linestyle='--', label='기준선 50%')
    ax.set_xlabel('FinBERT:Gemma 비율')
    ax.set_ylabel('평균 적중률 (%)')
    ax.set_title('가중치별 평균 적중률')
    ax.legend()
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()

else:
    # Gemma 미실행: FinBERT 단독 적중률만 시각화
    fig, ax = plt.subplots(figsize=(10, 5))
    lags = list(lag_cols.keys())
    accs = result_df['적중률(%)'].values
    colors_bar = ['#3498db' if a >= 50 else '#e74c3c' for a in accs]
    bars = ax.bar(lags, accs, color=colors_bar, alpha=0.8)
    ax.axhline(50, color='gray', linestyle='--', linewidth=1.5, label='기준선 50%')
    ax.set_xlabel('시간 지연')
    ax.set_ylabel('적중률 (%)')
    ax.set_title('뉴스 감성 → 비트코인 방향 예측 적중률 (FinBERT 기준)')
    ax.legend()
    ax.set_ylim(0, 100)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


---
## 9. 최적 가중치 결론

예측 적중률 기준으로 최적 FinBERT:Gemma 가중치를 선정한다.


In [ ]:
# ── 최적 가중치 선정 ─────────────────────────────────────────────────────
print('=' * 60)
print('  최적 가중치 분석 결과')
print('=' * 60)

if weight_hit_df is not None and not weight_hit_df.empty:
    avg_acc = weight_hit_df.mean(axis=1)
    best_weight = avg_acc.idxmax()
    best_acc = avg_acc.max()

    print(f'\n가중치별 평균 적중률:')
    for w, a in avg_acc.items():
        marker = ' ← 최적' if w == best_weight else ''
        print(f'  {w}: {a:.1f}%{marker}')

    # 최적 가중치 추출
    fw_pct = int(best_weight.split(':')[0].replace('F',''))
    gw_pct = int(best_weight.split(':')[1].replace('G',''))

    print(f'\n[결론]')
    print(f'  최적 가중치: FinBERT {fw_pct}% : Gemma {gw_pct}%')
    print(f'  평균 적중률: {best_acc:.1f}%  (기준선 50% 대비 +{best_acc-50:.1f}%p)')

    if best_acc - 50 < 5:
        print(f'\n  ⚠️  기준선 대비 개선폭이 5%p 미만입니다.')
        print(f'      샘플 수({len(weight_hit_df)}건)가 적어 통계적 유의성이 낮을 수 있습니다.')
        print(f'      더 많은 뉴스 데이터 수집 후 재분석을 권장합니다.')

    print(f'\n[.env 권장 설정]')
    print(f'  FINBERT_WEIGHT={fw_pct/100:.2f}')
    print(f'  GEMMA_WEIGHT={gw_pct/100:.2f}')

else:
    best_acc = result_df['적중률(%)'].max()
    best_lag = result_df['적중률(%)'].idxmax()
    print(f'\nGemma 미실행 — FinBERT 단독 결과')
    print(f'  최고 적중률: {best_acc:.1f}% ({best_lag} 기준)')
    print(f'  기준선 대비: +{best_acc-50:.1f}%p')
    print(f'\n  Gemma 실행 후 재분석하면 최적 가중치를 찾을 수 있습니다.')
    print(f'  Ollama 설치: https://ollama.ai')
    print(f'  모델 다운로드: ollama pull gemma3:4b')


---
## 10. 전체 적중률 요약

감성 분석 기반 BTC 가격 예측의 전반적인 성능을 종합 정리한다.


In [ ]:
# ── 전체 적중률 요약 ─────────────────────────────────────────────────────
print('=' * 65)
print('  뉴스 감성 분석 → 비트코인 가격 방향 예측 최종 요약')
print('=' * 65)

print(f'\n[데이터 현황]')
print(f'  분석 뉴스 기사   : {len(news_df)}건')
print(f'  가격 윈도우 수   : {len(ma)}개 (5분 단위)')
print(f'  데이터 기간      : {merged_df["window_start"].min().date()} ~ {merged_df["window_start"].max().date()}')

print(f'\n[예측 성능 요약]')
print(result_df[['적중률(%)', '정밀도(%)', '재현율(%)', 'F1(%)']].to_string())

best_acc_val = result_df['적중률(%)'].max()
best_acc_lag = result_df['적중률(%)'].idxmax()
worst_acc_val = result_df['적중률(%)'].min()
worst_acc_lag = result_df['적중률(%)'].idxmin()

print(f'\n[핵심 발견]')
print(f'  최고 적중률 : {best_acc_val:.1f}% ({best_acc_lag}) — 기준선 대비 +{best_acc_val-50:.1f}%p')
print(f'  최저 적중률 : {worst_acc_val:.1f}% ({worst_acc_lag}) — 기준선 대비 {worst_acc_val-50:+.1f}%p')
print(f'  평균 적중률 : {result_df["적중률(%)"].mean():.1f}%')

print(f'\n[감성-가격 관계 해석]')
if best_acc_val > 60:
    print(f'  ✅ {best_acc_lag} 기준으로 유의미한 예측력 확인 ({best_acc_val:.1f}%)')
    print(f'     뉴스 감성이 해당 시간대 BTC 가격 방향과 높은 상관 관계를 보임')
elif best_acc_val > 55:
    print(f'  🔶 {best_acc_lag} 기준으로 약한 예측력 확인 ({best_acc_val:.1f}%)')
    print(f'     단독 지표로는 약하지만, 다른 기술적 지표와 결합 시 유용할 수 있음')
else:
    print(f'  ⚠️  현재 샘플로는 기준선(50%)을 크게 상회하지 못합니다.')
    print(f'     원인: 1) 샘플 수 부족  2) 뉴스-가격 시간차 불일치  3) 외부 변수 영향')
    print(f'     권장: CryptoCompare body 포함 뉴스 수집 확대 후 재분석')

print(f'\n[다음 단계]')
print(f'  1. CryptoCompare 뉴스 수집 확대 (현재 {len(news_df)}건 → 목표 500건 이상)')
print(f'  2. Gemma 실행 후 최적 가중치 재분석')
print(f'  3. 감성 점수 임계값 조정 (현재 0 기준 → ROC curve로 최적값 탐색)')
print(f'  4. 시장 상황(상승장/하락장) 구분 후 조건부 분석')
